In [2]:
import pandas as pd

df = pd.read_csv("../data/raw/fake_reviews.csv")

df.head()

,category,rating,label,text_
0,Home_and_Kitchen_5,5.0,CG,"Love this! Well made, sturdy, and very comfor..."
1,Home_and_Kitchen_5,5.0,CG,"love it, a great upgrade from the original. I..."
2,Home_and_Kitchen_5,5.0,CG,This pillow saved my back. I love the look and...
3,Home_and_Kitchen_5,1.0,CG,"Missing information on how to use it, but it i..."
4,Home_and_Kitchen_5,5.0,CG,Very nice set. Good quality. We have had the s...


In [4]:
df.columns
df.head()

,category,rating,label,text_
0,Home_and_Kitchen_5,5.0,CG,"Love this! Well made, sturdy, and very comfor..."
1,Home_and_Kitchen_5,5.0,CG,"love it, a great upgrade from the original. I..."
2,Home_and_Kitchen_5,5.0,CG,This pillow saved my back. I love the look and...
3,Home_and_Kitchen_5,1.0,CG,"Missing information on how to use it, but it i..."
4,Home_and_Kitchen_5,5.0,CG,Very nice set. Good quality. We have had the s...


In [7]:
df['label'].value_counts()

label
CG    20216
OR    20216
Name: count, dtype: int64

In [9]:
df['review_length'] = df['text_'].apply(lambda x: len(str(x).split()))

df.groupby('label')['review_length'].mean()

label
CG    61.288237
OR    73.642610
Name: review_length, dtype: float64

In [10]:
df.groupby('label')['rating'].value_counts()

label  rating
CG     5.0       12319
       4.0        3920
       3.0        1952
       1.0        1063
       2.0         962
OR     5.0       12240
       4.0        4045
       3.0        1834
       1.0        1092
       2.0        1005
Name: count, dtype: int64

In [6]:
print(type(df))
print("FAKE REVIEWS:\n")
print(df[df['label'] == 'CG']['text_'].head(5))

print("\nREAL REVIEWS:\n")
print(df[df['label'] == 'OR']['text_'].head(5))

<class 'pandas.core.frame.DataFrame'>
FAKE REVIEWS:

0    Love this!  Well made, sturdy, and very comfor...
1    love it, a great upgrade from the original.  I...
2    This pillow saved my back. I love the look and...
3    Missing information on how to use it, but it i...
4    Very nice set. Good quality. We have had the s...
Name: text_, dtype: object

REAL REVIEWS:

55    These are just perfect, exactly what I was loo...
56    Such a great purchase can't beat it for the price
57    What can you say--- cheap and it works as inte...
58    These are so nice, sturdy, like the color choi...
59        It is nice bowl and have had a fast shipping!
Name: text_, dtype: object


In [12]:
df['exclamation_count'] = df['text_'].apply(lambda x: str(x).count('!'))

df.groupby('label')['exclamation_count'].mean()

label
CG    0.275574
OR    0.519490
Name: exclamation_count, dtype: float64

In [5]:
df['uppercase_count'] = df['text_'].apply(lambda x: sum(1 for c in str(x) if c.isupper()))

df.groupby('label')['uppercase_count'].mean()

label
CG     8.552879
OR    11.012268
Name: uppercase_count, dtype: float64

In [6]:
from collections import Counter

fake_words = Counter(" ".join(df[df['label']=='CG']['text_']).split()).most_common(10)
real_words = Counter(" ".join(df[df['label']=='OR']['text_']).split()).most_common(10)

print("Fake words:", fake_words)
print("Real words:", real_words)

Fake words: [('the', 61916), ('I', 56621), ('a', 56338), ('and', 43066), ('is', 40221), ('to', 30421), ('it', 24262), ('The', 21588), ('this', 20510), ('of', 20289)]
Real words: [('the', 63509), ('and', 43003), ('I', 38578), ('a', 38545), ('to', 37329), ('of', 24825), ('is', 23734), ('it', 21398), ('for', 17587), ('in', 17282)]


In [11]:
# Review length (already done)

# Exclamation marks
df['exclamation_count'] = df['text_'].apply(lambda x: str(x).count('!'))

# Uppercase count
df['uppercase_count'] = df['text_'].apply(lambda x: sum(1 for c in str(x) if c.isupper()))

# Word count (already similar but cleaner)
df['word_count'] = df['text_'].apply(lambda x: len(str(x).split()))

In [8]:
df[['text_', 'exclamation_count', 'uppercase_count', 'word_count']].head()

,text_,exclamation_count,uppercase_count,word_count
0,"Love this! Well made, sturdy, and very comfor...",2,4,12
1,"love it, a great upgrade from the original. I...",0,1,16
2,This pillow saved my back. I love the look and...,0,2,14
3,"Missing information on how to use it, but it i...",1,2,17
4,Very nice set. Good quality. We have had the s...,0,3,18


In [9]:
df.groupby('label')[['exclamation_count','uppercase_count','word_count']].mean()

,exclamation_count,uppercase_count,word_count
label,,,
CG,0.275574,8.552879,61.288237
OR,0.519490,11.012268,73.642610


In [ ]:
# Train/test split first so the test set is not used to fit TF-IDF.
from sklearn.model_selection import train_test_split

texts = df['text_'].fillna('').astype(str)
y = df['label'].map({'CG': 1, 'OR': 0})

train_text, test_text, y_train, y_test, train_idx, test_idx = train_test_split(
    texts, y, df.index, test_size=0.20, random_state=42, stratify=y
)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000, sublinear_tf=True)
X_text_train = tfidf.fit_transform(train_text)
X_text_test = tfidf.transform(test_text)


In [ ]:
import numpy as np
from scipy.sparse import hstack, csr_matrix

X_features = df[['exclamation_count', 'uppercase_count', 'word_count']].values.astype(float)
X_train = hstack([X_text_train, csr_matrix(X_features[train_idx])], format='csr')
X_test = hstack([X_text_test, csr_matrix(X_features[test_idx])], format='csr')


In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=2000, solver='liblinear', random_state=42)
model.fit(X_train, y_train)


In [ ]:
y_pred = model.predict(X_test)


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print('Accuracy:', accuracy_score(y_test, y_pred))
print('\nReport:\n', classification_report(y_test, y_pred, digits=4))
print('Confusion matrix:\n', confusion_matrix(y_test, y_pred))


In [ ]:
import pickle

with open('../model.pkl', 'wb') as f:
    pickle.dump(model, f)
with open('../tfidf.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

print('Model and TF-IDF vectorizer saved successfully.')
